# Rung 2 — CAT-Q + AYOT

Identical quantizer and hyperparameters to rung 1; only the calibration corpus changes to the model's own reasoning traces. Fires when rung 1 misses its gate.

## Setup

In [ ]:
# Colab Pro A100 setup. Persist results/checkpoints on Drive so a session
# expiry costs the run, not the artifacts.
from google.colab import drive
drive.mount("/content/drive")
REPO = "/content/drive/MyDrive/lfm3-ternary"
import os, subprocess, sys
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/ORG/LFM3-Ternary", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
os.makedirs("results", exist_ok=True)
!pip -q install "transformers>=5.12.1" "lm-eval==0.4.7" accelerate datasets sentencepiece safetensors gguf
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


## Self-generated traces (~4M tokens)

In [ ]:
!python3 scripts/build_calib_ayot.py --model LiquidAI/LFM3-1.7B \
    --out-dir data/calib/ayot --tokens 4000000 --sample-review 20
# review data/calib/ayot/review.jsonl before spending calibration hours;
# if the traces are junk, fall back to teacher traces and record the deviation in REPORT.md.

## Quantize

In [ ]:
# The rung runner writes the checkpoint and its ternary manifest; the eval
# script then produces the score file gates.py consumes.
!python3 scripts/run_catq.py --model LiquidAI/LFM3-1.7B --calib data/calib/ayot --out checkpoints/rung2 --epochs 60
!bash scripts/eval_lm_eval.sh checkpoints/rung2 results/rung2.json
!python3 scripts/gates.py tier0 --baseline results/baseline.json --scores results/rung2.json \
    --rung 2 --remaining 1 --out results/rung2_verdict.json
